# 2층 신경망 역전파 수식 유도

고정 예제는 입력 2개 → 은닉 2개 → 출력 1개 구조다. 모든 벡터는 NumPy의 1차원 배열로 표기한다.

| 기호 | shape | 값 |
|---|---:|---|
| $x$ | $(2,)$ | $[1,0]$ |
| $W_1$ | $(2,2)$ | $[[0.1,0.2],[0.3,0.4]]$ |
| $b_1$ | $(2,)$ | $[0,0]$ |
| $W_2$ | $(2,)$ | $[0.5,0.6]$ |
| $b_2$ | scalar | $0$ |
| $y$ | scalar | $1$ |

활성화는 $\sigma(t)=1/(1+e^{-t})$, 손실은 $L=-[y\log\hat y+(1-y)\log(1-\hat y)]$이다.

## 순전파 손계산

1. $z_1=W_1x+b_1$ — shape $(2,)$: $[0.1,0.3]$
2. $a_1=\sigma(z_1)$ — shape $(2,)$: $[0.5250,0.5744]$
3. $z_2=W_2^Ta_1+b_2$ — scalar: $0.5(0.5250)+0.6(0.5744)=0.6072$
4. $\hat y=\sigma(z_2)$ — scalar: $0.6473$
5. $L=-\log(\hat y)$ — scalar: $0.4350$

In [1]:
import numpy as np

np.random.seed(42)
x = np.array([1.0, 0.0])
W1 = np.array([[0.1, 0.2], [0.3, 0.4]])
b1 = np.array([0.0, 0.0])
W2 = np.array([0.5, 0.6])
b2 = 0.0
y_true = 1.0

def sigmoid(value):
    return 1.0 / (1.0 + np.exp(-value))

z1 = W1 @ x + b1
a1 = sigmoid(z1)
z2 = W2 @ a1 + b2
y_pred = sigmoid(z2)
loss = -(y_true * np.log(y_pred) + (1.0 - y_true) * np.log(1.0 - y_pred))

for name, value in [('z1', z1), ('a1', a1), ('z2', z2), ('y_pred', y_pred), ('L', loss)]:
    print(f'{name:8s} shape={np.asarray(value).shape or "scalar"!s:8s} value={np.round(value, 4)}')

z1       shape=(2,)     value=[0.1 0.3]
a1       shape=(2,)     value=[0.525  0.5744]
z2       shape=scalar   value=0.6072
y_pred   shape=scalar   value=0.6473
L        shape=scalar   value=0.435


## 역전파: 출력층에서 입력층 방향으로

BCE를 미분하면
$$\frac{\partial L}{\partial \hat y}=-\frac{y}{\hat y}+\frac{1-y}{1-\hat y}$$
이며 shape은 scalar, 값은 $-1.5449$다.

Sigmoid 미분 $\partial\hat y/\partial z_2=\hat y(1-\hat y)$를 곱하면
$$\frac{\partial L}{\partial z_2}=\frac{\partial L}{\partial\hat y}\hat y(1-\hat y)=\hat y-y=-0.3527$$
이고 shape은 scalar다.

$$\frac{\partial L}{\partial W_2}=\frac{\partial L}{\partial z_2}a_1=[-0.1852,-0.2026] \quad \text{shape }(2,)$$
$$\frac{\partial L}{\partial a_1}=\frac{\partial L}{\partial z_2}W_2=[-0.1764,-0.2116] \quad \text{shape }(2,)$$
$$\frac{\partial L}{\partial z_1}=\frac{\partial L}{\partial a_1}\odot a_1\odot(1-a_1)=[-0.0440,-0.0517] \quad \text{shape }(2,)$$
$$\frac{\partial L}{\partial W_1}=\frac{\partial L}{\partial z_1}x^T=\begin{bmatrix}-0.0440&0\\-0.0517&0\end{bmatrix} \quad \text{shape }(2,2)$$

참고로 편향 기울기는 $\partial L/\partial b_2=\partial L/\partial z_2$와 $\partial L/\partial b_1=\partial L/\partial z_1$이다.

In [2]:
dL_dy_pred = -(y_true / y_pred) + (1.0 - y_true) / (1.0 - y_pred)
dL_dz2 = y_pred - y_true
dL_dW2 = dL_dz2 * a1
dL_da1 = dL_dz2 * W2
dL_dz1 = dL_da1 * a1 * (1.0 - a1)
dL_dW1 = np.outer(dL_dz1, x)

gradients = [
    ('dL/dy_pred', dL_dy_pred),
    ('dL/dz2', dL_dz2),
    ('dL/dW2', dL_dW2),
    ('dL/da1', dL_da1),
    ('dL/dz1', dL_dz1),
    ('dL/dW1', dL_dW1),
]
for name, value in gradients:
    print(f'{name:12s} shape={np.asarray(value).shape or "scalar"!s:8s} value={np.round(value, 4)}')

dL/dy_pred   shape=scalar   value=-1.5449
dL/dz2       shape=scalar   value=-0.3527
dL/dW2       shape=(2,)     value=[-0.1852 -0.2026]
dL/da1       shape=(2,)     value=[-0.1764 -0.2116]
dL/dz1       shape=(2,)     value=[-0.044  -0.0517]
dL/dW1       shape=(2, 2)   value=[[-0.044  -0.    ]
 [-0.0517 -0.    ]]


## NumPy 검증

아래 기대값은 위 손계산에서 독립적으로 기록한 리터럴이다. NumPy 결과와 소수점 넷째 자리까지 비교한다.

In [3]:
expected = {
    'z1': np.array([0.1000, 0.3000]),
    'a1': np.array([0.5250, 0.5744]),
    'z2': np.array(0.6072),
    'y_pred': np.array(0.6473),
    'dL/dy_pred': np.array(-1.5449),
    'dL/dz2': np.array(-0.3527),
    'dL/dW2': np.array([-0.1852, -0.2026]),
    'dL/da1': np.array([-0.1764, -0.2116]),
    'dL/dz1': np.array([-0.0440, -0.0517]),
    'dL/dW1': np.array([[-0.0440, 0.0], [-0.0517, 0.0]]),
}
actual = {
    'z1': z1, 'a1': a1, 'z2': z2, 'y_pred': y_pred,
    'dL/dy_pred': dL_dy_pred, 'dL/dz2': dL_dz2,
    'dL/dW2': dL_dW2, 'dL/da1': dL_da1,
    'dL/dz1': dL_dz1, 'dL/dW1': dL_dW1,
}
for name, expected_value in expected.items():
    np.testing.assert_array_almost_equal(actual[name], expected_value, decimal=4)
print('모든 순전파·역전파 값이 손계산 결과와 소수점 4자리까지 일치합니다.')

모든 순전파·역전파 값이 손계산 결과와 소수점 4자리까지 일치합니다.
